# Solucao Completa: Modelos Ensemble com Qualidade de Vinhos

Esta solucao implementa uma comparacao reproduzivel entre um baseline e metodos de Bagging, Boosting, Stacking e Voting. O fluxo usa pipelines para que imputacao, codificacao e escala sejam ajustadas somente no treino.

## 1. Carregamento e analise inicial
A base possui atributos fisico-quimicos, uma coluna categorica (`lote`) e a nota de qualidade atribuida ao vinho. A nota sera usada somente para formar o alvo de classificacao.

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (
    AdaBoostClassifier, BaggingClassifier, ExtraTreesClassifier,
    GradientBoostingClassifier, RandomForestClassifier,
    StackingClassifier, VotingClassifier
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

SEED = 42
sns.set_theme(style='whitegrid')

df = pd.read_csv('vinhos.csv')
print('Dimensoes:', df.shape)
display(df.head())
display(df.isna().sum().sort_values(ascending=False))
print('Duplicatas:', df.duplicated().sum())
display(df['qualidade'].value_counts().sort_index())

## 2. Criacao do alvo e divisao dos dados
A categoria e derivada da nota: `ruim` para notas ate 5, `medio` para nota 6 e `bom` para notas a partir de 7. A nota original e excluida das features para impedir vazamento direto do alvo.

In [ ]:
df_model = df.drop_duplicates().copy()

def categorizar_qualidade(nota):
    if nota <= 5:
        return 'ruim'
    if nota == 6:
        return 'medio'
    return 'bom'

df_model['categoria'] = df_model['qualidade'].apply(categorizar_qualidade)
feature_cols = [coluna for coluna in df_model.columns if coluna not in ['qualidade', 'categoria']]
X = df_model[feature_cols]
y = df_model['categoria']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y
)

print('Treino:', X_train.shape, '| Teste:', X_test.shape)
display(y.value_counts(normalize=True).sort_index().rename('proporcao'))

## 3. Pre-processamento e funcao de avaliacao
Cada pipeline recebe um novo `ColumnTransformer`. Portanto, mediana, escala e categorias sao aprendidas dentro de cada ajuste, exclusivamente com os dados de treino.

In [ ]:
numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = ['lote']

def criar_preprocessador():
    numeric_transformer = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])
    categorical_transformer = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])
    return ColumnTransformer([
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

def criar_pipeline(modelo):
    return Pipeline([
        ('preprocessador', criar_preprocessador()),
        ('modelo', modelo)
    ])

resultados = []
modelos_treinados = {}

def avaliar_modelo(nome, modelo, categoria):
    inicio = time.perf_counter()
    modelo.fit(X_train, y_train)
    tempo = time.perf_counter() - inicio
    predicoes = modelo.predict(X_test)
    resultados.append({
        'Modelo': nome,
        'Categoria': categoria,
        'Acuracia': accuracy_score(y_test, predicoes),
        'F1 macro': f1_score(y_test, predicoes, average='macro'),
        'Tempo (s)': tempo
    })
    modelos_treinados[nome] = modelo
    print(f'{nome}: acuracia={resultados[-1]["Acuracia"]:.4f} | F1 macro={resultados[-1]["F1 macro"]:.4f} | tempo={tempo:.2f}s')

## 4. Baseline
A arvore de decisao isolada e um ponto de comparacao. Arvores profundas sao modelos instaveis: pequenas mudancas nos dados podem alterar bastante sua estrutura, motivo pelo qual Bagging costuma ajudar.

In [ ]:
baseline = criar_pipeline(DecisionTreeClassifier(random_state=SEED))
avaliar_modelo('Arvore de Decisao', baseline, 'Baseline')

## 5. Bagging: reducao de variancia
Bagging treina arvores em amostras bootstrap e agrega as previsoes. Random Forest adiciona aleatoriedade na selecao de atributos; Extra Trees introduz ainda mais aleatoriedade nas divisoes.

In [ ]:
bagging = criar_pipeline(BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=SEED),
    n_estimators=150, max_samples=0.8, max_features=0.8,
    random_state=SEED, n_jobs=-1
) )
random_forest = criar_pipeline(RandomForestClassifier(
    n_estimators=150, random_state=SEED, n_jobs=-1
) )
extra_trees = criar_pipeline(ExtraTreesClassifier(
    n_estimators=150, random_state=SEED, n_jobs=-1
) )

avaliar_modelo('Bagging', bagging, 'Bagging')
avaliar_modelo('Random Forest', random_forest, 'Bagging')
avaliar_modelo('Extra Trees', extra_trees, 'Bagging')

## 6. Importancia das variaveis no Random Forest
A importancia por impureza e uma medida descritiva do modelo. Ela nao estabelece causalidade e pode favorecer atributos com mais possibilidades de divisao.

In [ ]:
preprocessador_rf = random_forest.named_steps['preprocessador']
nomes_atributos = preprocessador_rf.get_feature_names_out()
importancias = pd.Series(
    random_forest.named_steps['modelo'].feature_importances_,
    index=nomes_atributos
).sort_values()

plt.figure(figsize=(9, 6))
importancias.plot(kind='barh', color='steelblue')
plt.title('Importancia das variaveis - Random Forest')
plt.xlabel('Importancia por impureza')
plt.tight_layout()
plt.show()

## 8. Stacking e Voting
Stacking usa predicoes geradas por validacao cruzada dos modelos base como entrada para um meta-modelo. Voting agrega votos ou probabilidades; na variante soft, a decisao usa a media das probabilidades previstas.

Os valores de acuracia e F1 macro abaixo sao apenas um placar para acompanhar os palpites. Imagine a acuracia como a quantidade de acertos em um jogo e o F1 macro como uma regra que da o mesmo peso a cada categoria do jogo.

In [ ]:
adaboost = criar_pipeline(AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=2, random_state=SEED),
    n_estimators=150, learning_rate=0.05, random_state=SEED
) )
gradient_boosting = criar_pipeline(GradientBoostingClassifier(
    n_estimators=150, learning_rate=0.05, max_depth=3,
    subsample=0.8, random_state=SEED
) )

avaliar_modelo('AdaBoost', adaboost, 'Boosting')
avaliar_modelo('Gradient Boosting', gradient_boosting, 'Boosting')

## 8. Stacking e Voting
Stacking usa predicoes geradas por validacao cruzada dos modelos base como entrada para um meta-modelo. Voting agrega votos ou probabilidades; na variante soft, a decisao usa a media das probabilidades previstas.

In [ ]:
estimadores_stacking = [
    ('rf', RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1)),
    ('knn', KNeighborsClassifier(n_neighbors=9)),
    ('lr', LogisticRegression(max_iter=1000, random_state=SEED))
]
stacking = criar_pipeline(StackingClassifier(
    estimators=estimadores_stacking,
    final_estimator=LogisticRegression(max_iter=1000, random_state=SEED),
    cv=5, n_jobs=-1
) )

estimadores_voting = [
    ('rf', RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1)),
    ('gb', GradientBoostingClassifier(n_estimators=100, learning_rate=0.05, random_state=SEED)),
    ('lr', LogisticRegression(max_iter=1000, random_state=SEED))
]
voting_hard = criar_pipeline(VotingClassifier(estimators=estimadores_voting, voting='hard', n_jobs=-1))
voting_soft = criar_pipeline(VotingClassifier(estimators=estimadores_voting, voting='soft', n_jobs=-1))

avaliar_modelo('Stacking', stacking, 'Stacking')
avaliar_modelo('Hard Voting', voting_hard, 'Voting')
avaliar_modelo('Soft Voting', voting_soft, 'Voting')

## 9. Placar do experimento
Acuracia e F1 macro serao usadas somente para acompanhar os palpites de cada modelo. O tempo de treino e o tempo de escoragem completam a observacao: um modelo pode aprender devagar e ainda assim responder rapidamente, ou o contrario.

Uma comparacao real exigiria repeticoes, validacao adequada e consideracoes do problema. Aqui, o objetivo e enxergar as diferencas entre as familias de ensemble.

In [ ]:
df_resultados = pd.DataFrame(resultados).sort_values('F1 macro', ascending=False).reset_index(drop=True)
display(df_resultados.style.format({
    'Acuracia': '{:.4f}', 'F1 macro': '{:.4f}', 'Tempo (s)': '{:.2f}'
}))

plt.figure(figsize=(10, 5))
sns.barplot(data=df_resultados, x='F1 macro', y='Modelo', hue='Categoria', dodge=False)
plt.xlim(0, 1)
plt.title('Comparacao de modelos por F1 macro')
plt.tight_layout()
plt.show()

melhor_nome = df_resultados.iloc[0]['Modelo']
melhor_modelo = modelos_treinados[melhor_nome]
print('Melhor modelo por F1 macro:', melhor_nome)
ConfusionMatrixDisplay.from_estimator(melhor_modelo, X_test, y_test, cmap='Blues')
plt.title(f'Matriz de confusao - {melhor_nome}')
plt.tight_layout()
plt.show()

## Conclusao
A comparacao mostra que ensembles nao sao automaticamente superiores em toda metrica ou contexto. Bagging tende a reduzir variancia de arvores instaveis, Boosting busca reduzir vies de forma sequencial, e Stacking/Voting podem explorar a diversidade entre modelos. A decisao final deve considerar F1 macro, matriz de confusao, tempo de treino, interpretabilidade e requisitos de manutencao.

## 10. Desafio extra: Random Forest, LightGBM e CatBoost

A base `bank-vf.csv` descreve contatos de uma campanha de marketing bancario. Ela possui varias colunas categoricas e o alvo `y`, que indica se o cliente aceitou a oferta.

Nesta comparacao, Random Forest e LightGBM usam One-Hot Encoding. CatBoost recebe diretamente as colunas categoricas, uma capacidade nativa desse algoritmo. Usamos uma amostra fixa de 12 mil registros para que o relogio do experimento seja prático.

> Caso os pacotes ainda nao estejam no kernel, execute antes: `%pip install lightgbm catboost`.

In [ ]:
from sklearn.preprocessing import FunctionTransformer
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# Remove uma coluna de indice eventualmente gravada no CSV.
df_bank = pd.read_csv('bank-vf.csv')
colunas_indice = [coluna for coluna in df_bank.columns if str(coluna).lower().startswith('unnamed')]
df_bank = df_bank.drop(columns=colunas_indice)
df_bank = df_bank.dropna(subset=['y']).sample(n=min(12000, len(df_bank)), random_state=SEED).copy()

y_bank = df_bank['y'].map({'no': 0, 'yes': 1})
X_bank = df_bank.drop(columns='y')
X_bank_train, X_bank_test, y_bank_train, y_bank_test = train_test_split(
    X_bank, y_bank, test_size=0.25, random_state=SEED, stratify=y_bank
)

numeric_bank = X_bank.select_dtypes(include=np.number).columns.tolist()
categorical_bank = X_bank.select_dtypes(exclude=np.number).columns.tolist()

print('Treino:', X_bank_train.shape, '| Teste:', X_bank_test.shape)
print('Colunas categoricas:', categorical_bank)
print('Proporcao de aceite:', round(y_bank.mean(), 3))

### Pipelines e relogio

O mesmo relogio mede duas etapas: quanto tempo o modelo leva para aprender com os exemplos e quanto leva para gerar palpites novos. O pequeno placar de acuracia e F1 macro serve somente para acompanhar se os tres participantes estao jogando a mesma tarefa de forma razoavel.

In [ ]:
def criar_preprocessador_bank():
    return ColumnTransformer([
        ('num', SimpleImputer(strategy='median'), numeric_bank),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore'))
        ]), categorical_bank)
    ])

def preparar_catboost(dados):
    dados = dados.copy()
    for coluna in categorical_bank:
        dados[coluna] = dados[coluna].fillna('__ausente__').astype(str)
    return dados

modelos_bank = {
    'Random Forest': Pipeline([
        ('preprocessador', criar_preprocessador_bank()),
        ('modelo', RandomForestClassifier(n_estimators=150, random_state=SEED, n_jobs=-1))
    ]),
    'LightGBM': Pipeline([
        ('preprocessador', criar_preprocessador_bank()),
        ('modelo', LGBMClassifier(
            n_estimators=150, learning_rate=0.08, max_depth=6,
            random_state=SEED, n_jobs=-1, verbosity=-1
        ))
    ]),
    'CatBoost': Pipeline([
        ('preprocessador', FunctionTransformer(preparar_catboost, validate=False)),
        ('modelo', CatBoostClassifier(
            iterations=150, depth=6, learning_rate=0.08, random_state=SEED,
            cat_features=categorical_bank, verbose=False, thread_count=-1
        ))
    ])
}

def medir_modelo_bank(nome, modelo):
    inicio_treino = time.perf_counter()
    modelo.fit(X_bank_train, y_bank_train)
    tempo_treino = time.perf_counter() - inicio_treino

    inicio_escoragem = time.perf_counter()
    predicoes = modelo.predict(X_bank_test)
    tempo_escoragem = time.perf_counter() - inicio_escoragem

    return {
        'Modelo': nome,
        'Acuracia (placar)': accuracy_score(y_bank_test, predicoes),
        'F1 macro (placar)': f1_score(y_bank_test, predicoes, average='macro'),
        'Tempo de treino (s)': tempo_treino,
        'Tempo de escoragem (s)': tempo_escoragem
    }

resultados_bank = [
    medir_modelo_bank(nome, modelo)
    for nome, modelo in modelos_bank.items()
]
df_resultados_bank = pd.DataFrame(resultados_bank).sort_values('Tempo de treino (s)').reset_index(drop=True)
display(df_resultados_bank.style.format({
    'Acuracia (placar)': '{:.3f}',
    'F1 macro (placar)': '{:.3f}',
    'Tempo de treino (s)': '{:.3f}',
    'Tempo de escoragem (s)': '{:.3f}'
}))

ax = df_resultados_bank.set_index('Modelo')[['Tempo de treino (s)', 'Tempo de escoragem (s)']].plot(
    kind='bar', figsize=(9, 4), color=['steelblue', 'darkorange']
)
ax.set_title('Relogio do experimento - base bancaria')
ax.set_ylabel('Segundos')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()